# IMPORTS & CONFIG

In [ ]:
# =============================================================================
# CELL 1 — IMPORTS & CONFIG
# =============================================================================
# LLM Query Layer over VLM-adjusted driver rankings.
# Model : qwen2.5:14b via Ollama (local)
# Temp  : 0.1 — low temperature for consistent, repeatable answers
# Seed  : 42  — fixed seed for reproducible manager question results
# =============================================================================

import os
import re
import ast
import json
import math
import time
import random
import requests
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS_DIR    = "Outputs"
ANON_CLEAN_DIR = "debug_frames_qwen_anon_clean"

# ── Ollama config ─────────────────────────────────────────────────────────────
OLLAMA_BASE_URL = "http://localhost:" #make sure to add your own local host
OLLAMA_MODEL    = "qwen2.5:14b"
OLLAMA_TEMP     = 0.1
OLLAMA_SEED     = 42

# ── Random seed for reproducible manager question vehicle sampling ─────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Colour palette ────────────────────────────────────────────────────────────
C_BASE = "#2196F3"
C_VLM  = "#FF5722"
C_DIFF = "#4CAF50"

print("✓ Config loaded")
print(f"  Model    : {OLLAMA_MODEL}")
print(f"  Temp     : {OLLAMA_TEMP}")
print(f"  Seed     : {OLLAMA_SEED}")
print(f"  Outputs  : {os.path.abspath(OUTPUTS_DIR)}")

# LOAD ALL OUTPUTS

In [ ]:
# =============================================================================
# CELL 2 — LOAD ALL OUTPUTS
# =============================================================================

# ── Rankings ──────────────────────────────────────────────────────────────────
final_monthly_rankings_df = pd.read_csv(
    os.path.join(OUTPUTS_DIR, "final_monthly_rankings_vlm.csv")
)
final_monthly_rankings_df["window_start"] = pd.to_datetime(
    final_monthly_rankings_df["window_start"], errors="coerce"
)

scored_monthly_vlm = pd.read_csv(
    os.path.join(OUTPUTS_DIR, "scored_monthly_vlm.csv")
)
scored_monthly_vlm["window_start"] = pd.to_datetime(
    scored_monthly_vlm["window_start"], errors="coerce"
)

# ── Features monthly — needed for per-fleet re-ranking ───────────────────────
features_monthly_vlm = pd.read_csv(
    os.path.join(OUTPUTS_DIR, "features_monthly_vlm.csv")
)
features_monthly_vlm["window_start"] = pd.to_datetime(
    features_monthly_vlm["window_start"], errors="coerce"
)

# ── Event change audit ────────────────────────────────────────────────────────
event_change_audit_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "event_change_audit.csv"))

# ── Fused event df ────────────────────────────────────────────────────────────
fused_event_df = pd.read_csv(
    os.path.join(OUTPUTS_DIR, "fused_event_df.csv"),
    dtype={"terminal_event_id": str}
)

for col in ["det_codes", "rev_codes", "det_event_descriptions", "rev_event_descriptions",
            "det_rankable_event_descriptions", "rev_rankable_event_descriptions",
            "extra_rankable_vlm_events"]:
    if col in fused_event_df.columns:
        fused_event_df[col] = fused_event_df[col].apply(
            lambda x: ast.literal_eval(x)
            if isinstance(x, str) and x.startswith("[") else
            (x if isinstance(x, list) else [])
        )

# ── Covered ratio ─────────────────────────────────────────────────────────────
covered_ratio_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "covered_ratio.csv"))

# ── All vehicle IDs in fleet ──────────────────────────────────────────────────
all_vehicle_ids = sorted(
    final_monthly_rankings_df["vehicle_id"].dropna().astype("int64").unique().tolist()
)

print(f"final_monthly_rankings_vlm : {final_monthly_rankings_df.shape}")
print(f"scored_monthly_vlm         : {scored_monthly_vlm.shape}")
print(f"features_monthly_vlm       : {features_monthly_vlm.shape}")
print(f"event_change_audit         : {event_change_audit_df.shape}")
print(f"fused_event_df             : {fused_event_df.shape}")
print(f"covered_ratio              : {covered_ratio_df.shape}")
print(f"Total fleet vehicles       : {len(all_vehicle_ids)}")

# OLLAMA HELPERS

In [ ]:
# =============================================================================
# CELL 3 — OLLAMA HELPERS
# =============================================================================

def check_ollama():
    """Check Ollama is running and model is available."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        if OLLAMA_MODEL in models:
            print(f"✓ Ollama running — {OLLAMA_MODEL} available")
        else:
            print(f"⚠ Ollama running but {OLLAMA_MODEL} not found")
            print(f"  Available models: {models}")
            print(f"  Run: ollama pull {OLLAMA_MODEL}")
    except Exception as e:
        print(f"✗ Ollama not reachable: {e}")
        print(f"  Start with: ollama serve")

def ask_llm(prompt: str, system: str = "", max_tokens: int = 2048) -> tuple[str, float]:
    """
    Send a prompt to Ollama and return (response_text, elapsed_seconds).
    Temperature 0.1 and fixed seed for reproducibility.
    """
    payload = {
        "model"  : OLLAMA_MODEL,
        "prompt" : prompt,
        "stream" : False,
        "options": {
            "temperature" : OLLAMA_TEMP,
            "seed"        : OLLAMA_SEED,
            "num_predict" : max_tokens,
        },
    }
    if system:
        payload["system"] = system

    t0 = time.perf_counter()
    r  = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json=payload,
        timeout=1500,
    )
    elapsed = time.perf_counter() - t0
    r.raise_for_status()
    return r.json().get("response", "").strip(), elapsed

def print_llm_response(response: str, elapsed: float, title: str = "LLM Response"):
    """Display LLM response with timing."""
    print(f"\n{'='*70}")
    print(f"{title}  [{elapsed:.1f}s]")
    print(f"{'='*70}")
    display(Markdown(response))

check_ollama()

# CONTEXT BUILDERS

In [ ]:
# =============================================================================
# CELL 4 — CORE DATA & HELPERS
# =============================================================================

INSUFFICIENT_EXPOSURE_HOURS = 3.0
MIN_CLIPS_FOR_CONFIDENCE    = 3      # below this = limited visual evidence
MAX_CLIPS_OVERVIEW          = 0      # fleet overview never shows clips
MAX_CLIPS_INVESTIGATION     = 5      # investigation shows top N vehicles clips
MAX_CLIPS_DEEP_DIVE         = 10    # deep dive shows all clips

def _rank01(s: pd.Series) -> pd.Series:
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return s.rank(method="average", pct=True).fillna(0.0)

def add_score_and_class(df_in: pd.DataFrame, time_col: str) -> pd.DataFrame:
    d = df_in.copy()
    if "window_end" not in d.columns:
        d["window_end"] = d[time_col] + pd.Timedelta(days=1)
    d["r_harsh_total"]       = _rank01(d["harsh_episode_count"])
    d["r_short_speeding"]    = _rank01(d["short_speeding_episode_count"])
    d["r_long_speeding"]     = _rank01(d["long_speeding_episode_count"])
    d["r_power_total"]       = _rank01(d["power_violation_episode_count"])
    d["r_camera_total"]      = _rank01(d["camera_obstruction_episode_count"])
    d["r_fatigue_total"]     = _rank01(d["fatigue_episode_count"])
    d["r_distraction_total"] = _rank01(d["driver_distraction_episode_count"])
    d["r_situational_total"] = _rank01(d["situational_risk_episode_count"])
    d["ubi_proxy_score"] = (
        0.20 * d["r_harsh_total"]       + 0.12 * d["r_short_speeding"]
      + 0.18 * d["r_long_speeding"]     + 0.10 * d["r_power_total"]
      + 0.05 * d["r_camera_total"]      + 0.12 * d["r_fatigue_total"]
      + 0.13 * d["r_distraction_total"] + 0.10 * d["r_situational_total"]
    )
    d["insufficient_exposure_flag"] = (
        d["drive_hours"].fillna(0.0) < INSUFFICIENT_EXPOSURE_HOURS
    ).astype("int64")
    ranked_mask = d["insufficient_exposure_flag"] == 0
    q1 = d.loc[ranked_mask, "ubi_proxy_score"].quantile(0.33) if ranked_mask.any() else 0.0
    q2 = d.loc[ranked_mask, "ubi_proxy_score"].quantile(0.67) if ranked_mask.any() else 0.0
    def lab(row):
        if row["insufficient_exposure_flag"] == 1: return "Insufficient Exposure"
        x = row["ubi_proxy_score"]
        if x <= q1: return "Low"
        if x <= q2: return "Medium"
        return "High"
    d["vehicle_behaviour_class"] = d.apply(lab, axis=1)
    keep = ["vehicle_id", time_col, "window_end", "drive_hours", "distance_km",
            "insufficient_exposure_flag", "harsh_episode_count",
            "short_speeding_episode_count", "long_speeding_episode_count",
            "speeding_episode_count", "power_violation_episode_count",
            "camera_obstruction_episode_count", "fatigue_episode_count",
            "driver_distraction_episode_count", "situational_risk_episode_count",
            "crash_count", "ubi_proxy_score", "vehicle_behaviour_class"]
    keep = [c for c in keep if c in d.columns]
    return d[keep].sort_values(["vehicle_id", time_col]).reset_index(drop=True)

def rerank_for_fleet(vehicle_ids: list) -> pd.DataFrame:
    ids = [int(v) for v in vehicle_ids]
    fleet_features = features_monthly_vlm[features_monthly_vlm["vehicle_id"].isin(ids)].copy()
    if fleet_features.empty:
        return pd.DataFrame()
    rescored = add_score_and_class(fleet_features, "window_start")
    rescored["fleet_rank"] = (
        rescored.groupby("window_start")["ubi_proxy_score"]
        .rank(method="dense", ascending=False).astype(int)
    )
    rescored["fleet_size"] = len(ids)
    return rescored.sort_values("fleet_rank").reset_index(drop=True)

def get_vehicle_summary(vehicle_ids: list) -> pd.DataFrame:
    ids = [int(v) for v in vehicle_ids]
    fleet_scored = rerank_for_fleet(ids)
    if fleet_scored.empty:
        return pd.DataFrame()
    fleet_scored["vehicle_id"] = fleet_scored["vehicle_id"].astype("int64")

    fleet_agg = (
        fleet_scored.groupby("vehicle_id")
        .agg(
            fleet_rank              =("fleet_rank",                       "min"),
            ubi_proxy_score         =("ubi_proxy_score",                  "mean"),
            vehicle_behaviour_class =("vehicle_behaviour_class",          "first"),
            drive_hours             =("drive_hours",                      "sum"),
            distance_km             =("distance_km",                      "sum"),
            harsh_episodes          =("harsh_episode_count",              "sum"),
            short_speeding_episodes =("short_speeding_episode_count",     "sum"),
            long_speeding_episodes  =("long_speeding_episode_count",      "sum"),
            power_violations        =("power_violation_episode_count",    "sum"),
            camera_obstruction      =("camera_obstruction_episode_count", "sum"),
            fatigue_episodes        =("fatigue_episode_count",            "sum"),
            distraction_episodes    =("driver_distraction_episode_count", "sum"),
            situational_risk        =("situational_risk_episode_count",   "sum"),
            crash_count             =("crash_count",                      "sum"),
        )
        .reset_index()
    )

    # Add cam_covered_review_flag for tier 2 clips (rev=[67])
    vid_fused = fused_event_df[fused_event_df["vehicle_id"].isin(ids)].copy()
    vid_fused["cam_covered_review_flag"] = vid_fused["rev_codes"].apply(
        lambda x: 1 if isinstance(x, list) and 67 in x else 0
    )

    det_counts = (
        vid_fused
        .groupby("vehicle_id")
        .agg(
            total_clips                  =("clip",                    "count"),
            phone_detections             =("phone_flag",              "sum"),
            seatbelt_detections          =("seatbelt_off_flag",       "sum"),
            fatigue_detections           =("fatigue_flag",            "sum"),
            distraction_detections       =("distraction_flag",        "sum"),
            smoking_detections           =("smoke_flag",              "sum"),
            cam_covered_detections       =("cam_covered_flag",        "sum"),
            cam_covered_review_detections=("cam_covered_review_flag", "sum"),
        )
        .reset_index()
    )
    det_counts["vehicle_id"] = det_counts["vehicle_id"].astype("int64")

    summary = fleet_agg.merge(det_counts, on="vehicle_id", how="left")
    summary = summary.merge(
        covered_ratio_df[["vehicle_id","covered_ratio","covered_clips"]],
        on="vehicle_id", how="left"
    ).fillna(0)

    # Per-hour rates
    h = summary["drive_hours"].replace(0, np.nan)
    summary["phone_per_hour"]              = (summary["phone_detections"]       / h).round(3).fillna(0)
    summary["seatbelt_per_hour"]           = (summary["seatbelt_detections"]    / h).round(3).fillna(0)
    summary["fatigue_per_hour"]            = (summary["fatigue_detections"]     / h).round(3).fillna(0)
    summary["harsh_per_hour"]              = (summary["harsh_episodes"]         / h).round(3).fillna(0)
    summary["distraction_per_hour"]        = (summary["distraction_episodes"]   / h).round(3).fillna(0)
    summary["short_speeding_per_hour"]     = (summary["short_speeding_episodes"]/ h).round(3).fillna(0)
    summary["long_speeding_per_hour"]      = (summary["long_speeding_episodes"] / h).round(3).fillna(0)
    summary["smoking_per_hour"]            = (summary["smoking_detections"]     / h).round(3).fillna(0)
    summary["speeding_total"]              = summary["short_speeding_episodes"] + summary["long_speeding_episodes"]
    summary["speeding_per_hour"]           = (summary["speeding_total"]         / h).round(3).fillna(0)

    # Round for cleanliness
    summary["drive_hours"]      = summary["drive_hours"].round(2)
    summary["ubi_proxy_score"]  = summary["ubi_proxy_score"].round(2)

    # Confidence flag
    summary["evidence_confidence"] = summary["total_clips"].apply(
        lambda x: "HIGH" if x >= MIN_CLIPS_FOR_CONFIDENCE else "LOW"
    )

    return summary.sort_values("fleet_rank").reset_index(drop=True)

def get_fleet_averages(summary_df: pd.DataFrame) -> dict:
    """Pre-calculate fleet averages for peer comparison."""
    total_h = summary_df["drive_hours"].sum()
    if total_h <= 0:
        return {}
    return {
        "phone_per_hour"       : round(summary_df["phone_detections"].sum()        / total_h, 3),
        "seatbelt_per_hour"    : round(summary_df["seatbelt_detections"].sum()      / total_h, 3),
        "fatigue_per_hour"     : round(summary_df["fatigue_detections"].sum()       / total_h, 3),
        "harsh_per_hour"       : round(summary_df["harsh_episodes"].sum()           / total_h, 3),
        "distraction_per_hour" : round(summary_df["distraction_episodes"].sum()     / total_h, 3),
        "smoking_per_hour"     : round(summary_df["smoking_detections"].sum()       / total_h, 3),
        "short_speeding_per_hour": round(summary_df["short_speeding_episodes"].sum()/ total_h, 3),
        "long_speeding_per_hour" : round(summary_df["long_speeding_episodes"].sum() / total_h, 3),
    }

def get_daily_trend(vehicle_id: int) -> pd.DataFrame:
    """
    Daily breakdown of violations for a single vehicle.
    Uses raw fused_event_df grouped by day.
    Note: 7 days of data — trend is indicative only.
    """
    vid_clips = fused_event_df[fused_event_df["vehicle_id"] == vehicle_id].copy()
    if vid_clips.empty or "event_ts_final" not in vid_clips.columns:
        return pd.DataFrame()
    vid_clips["day"] = pd.to_datetime(
        vid_clips["event_ts_final"], utc=True, errors="coerce"
    ).dt.date
    daily = (
        vid_clips.groupby("day")
        .agg(
            clips              =("clip",              "count"),
            phone_detections   =("phone_flag",        "sum"),
            seatbelt_violations=("seatbelt_off_flag", "sum"),
            fatigue_detections =("fatigue_flag",      "sum"),
            cam_covered        =("cam_covered_flag",  "sum"),
        )
        .reset_index()
        .sort_values("day")
    )
    return daily

def get_uncertain_clips(vehicle_id: int) -> pd.DataFrame:
    """Clips where VLM was mostly uncertain — used for calibration case."""
    uncertain_cols = [
        "seatbelt_uncertain_flag","phone_uncertain_flag",
        "distraction_uncertain_flag","fatigue_uncertain_flag","smoking_uncertain_flag"
    ]
    available = [c for c in uncertain_cols if c in fused_event_df.columns]
    if not available:
        return pd.DataFrame()
    vid_clips = fused_event_df[fused_event_df["vehicle_id"] == vehicle_id].copy()
    vid_clips["uncertain_count"] = vid_clips[available].sum(axis=1)
    return (
        vid_clips[vid_clips["uncertain_count"] > 0]
        [["clip","vehicle_id","uncertain_count","driver_vis","seatbelt","phone",
          "distraction","fatigue","smoking","why"]]
        .sort_values("uncertain_count", ascending=False)
        .reset_index(drop=True)
    )

def get_phone_clips(vehicle_id: int) -> pd.DataFrame:
    return (
        fused_event_df[
            (fused_event_df["vehicle_id"] == vehicle_id) &
            (fused_event_df["phone_flag"] == 1)
        ]
        [["clip","vehicle_id","event_ts_final","original_event_description",
          "why","phone","seatbelt","distraction"]]
        .sort_values("clip").reset_index(drop=True)
    )

def get_camera_covered_clips(vehicle_id: int) -> pd.DataFrame:
    return (
        fused_event_df[
            (fused_event_df["vehicle_id"] == vehicle_id) &
            (fused_event_df["cam_covered_flag"] == 1)
        ]
        [["clip","vehicle_id","event_ts_final","original_event_description","cam_s","why"]]
        .sort_values("clip").reset_index(drop=True)
    )

def get_behaviour_clips(vehicle_id: int, flag: str) -> pd.DataFrame:
    """Get all clips for a specific behaviour flag for IR evidence."""
    cols = ["clip","vehicle_id","event_ts_final","original_event_description","why",
            "seatbelt","phone","distraction","fatigue","smoking","driver_vis"]
    cols = [c for c in cols if c in fused_event_df.columns]
    return (
        fused_event_df[
            (fused_event_df["vehicle_id"] == vehicle_id) &
            (fused_event_df[flag] == 1)
        ][cols]
        .sort_values("clip").reset_index(drop=True)
    )

def build_main_fleet_overview(summary_df: pd.DataFrame) -> str:
    """
    Fleet overview as JSON for main LLM. Always used regardless of fleet size.
    /hr rates with human-readable form. No raw totals where rate exists.
    """
    def to_human(per_hour: float) -> str:
        if per_hour <= 0:
            return "none"
        h = 1.0 / per_hour
        if h < 0.1:
            return f"{per_hour:.2f}/hr ~multiple/min"
        if h < 1.5:
            return f"{per_hour:.2f}/hr ~multiple/hr"
        return f"{per_hour:.2f}/hr ~1 per {h:.1f}h"

    records = []
    for _, row in summary_df.iterrows():
        records.append({
            "vehicle_id"         : int(row["vehicle_id"]),
            "rank"               : int(row["fleet_rank"]),
            "class"              : str(row["vehicle_behaviour_class"]),
            "ubi_score"          : float(row["ubi_proxy_score"]),
            "drive_hours"        : float(row["drive_hours"]),
            "crashes"            : int(row.get("crash_count", 0)),
            "phone"              : to_human(float(row.get("phone_per_hour", 0))),
            "fatigue"            : to_human(float(row.get("fatigue_per_hour", 0))),
            "harsh"              : to_human(float(row.get("harsh_per_hour", 0))),
            "short_speeding"     : to_human(float(row.get("short_speeding_per_hour", 0))),
            "long_speeding"      : to_human(float(row.get("long_speeding_per_hour", 0))),
            "distraction"        : to_human(float(row.get("distraction_per_hour", 0))),
            "smoking"            : to_human(float(row.get("smoking_per_hour", 0))),
            "seatbelt"           : to_human(float(row.get("seatbelt_per_hour", 0))),
            "power_violations"   : int(row.get("power_violations", 0)),
            "situational_risk"   : int(row.get("situational_risk", 0)),
            "cam_covered_ratio"  : f"{float(row.get('covered_ratio', 0)):.1%}",
            "cam_covered_review" : int(row.get("cam_covered_review_detections", 0)),
            "total_clips"        : int(row.get("total_clips", 0)),
            "confidence"         : str(row.get("evidence_confidence", "UNKNOWN")),
        })
    return json.dumps(records, indent=2)

def find_clip_dir(clip_no: int) -> str:
    clip_name = f"clip_{int(clip_no):03d}"
    for sub in os.listdir(ANON_CLEAN_DIR):
        candidate = os.path.join(ANON_CLEAN_DIR, sub, clip_name)
        if os.path.isdir(candidate):
            return candidate
    return None

def show_clip_frames(clip_no: int, title: str = "", max_frames: int = 12):
    clip_dir = find_clip_dir(clip_no)
    if clip_dir is None:
        print(f"  clip_{int(clip_no):03d}: not found")
        return
    frame_paths = sorted(Path(clip_dir).glob("frame_driver_*.jpg"))[:max_frames]
    if not frame_paths:
        print(f"  clip_{int(clip_no):03d}: no frames found")
        return
    n_cols = min(len(frame_paths), 6)
    n_rows = math.ceil(len(frame_paths) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for i, ax in enumerate(axes_flat):
        if i < len(frame_paths):
            try:
                ax.imshow(mpimg.imread(str(frame_paths[i])))
                ax.set_title(frame_paths[i].stem, fontsize=7)
            except Exception:
                ax.set_facecolor("#222")
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=8, y=1.01)
    plt.tight_layout(pad=0.3)
    plt.show()

def show_evidence_log(clips_df: pd.DataFrame, vehicle_id: int,
                      behaviour: str, max_clips: int = 999):
    """Show IR-style evidence log with timestamps and VLM reasoning before frames."""
    if clips_df.empty:
        print(f"  No {behaviour} clips found for vehicle {vehicle_id}")
        return
    shown = clips_df.head(max_clips)
    print(f"\nEVIDENCE LOG — Vehicle {vehicle_id} | {behaviour} | {len(shown)} clips")
    print(f"{'─'*70}")
    print(f"{'Clip':<12} {'Timestamp':<28} {'VLM Reasoning'}")
    print(f"{'─'*70}")
    for _, row in shown.iterrows():
        clip_no   = int(row["clip"])
        ts        = str(row.get("event_ts_final", ""))[:19]
        why       = str(row.get("why", ""))[:60]
        print(f"clip_{clip_no:03d}   {ts:<28} {why}")
    print(f"{'─'*70}")
    for _, row in shown.iterrows():
        clip_no = int(row["clip"])
        ts      = str(row.get("event_ts_final",""))[:19]
        why     = str(row.get("why",""))
        print(f"\nclip_{clip_no:03d}  |  Vehicle {vehicle_id}  |  {behaviour}  |  {ts}  |  {why}")
        show_clip_frames(
            clip_no,
            title=f"Vehicle {vehicle_id}  |  clip_{clip_no:03d}  |  {behaviour}  |  {ts}",
            max_frames=12
        )

def summary_to_text(summary_df: pd.DataFrame, fleet_avgs: dict = None, columns: list = None) -> str:
    """Converts summary df to JSON string for LLM consumption. Filters to planner-selected columns if provided."""
    records = []
    fleet_size = len(summary_df)

    def to_human(per_hour: float) -> str:
        if per_hour <= 0:
            return "none"
        h = 1.0 / per_hour
        if h < 0.1:
            return f"{per_hour:.2f}/hr ~multiple/min"
        if h < 1.5:
            return f"{per_hour:.2f}/hr ~multiple/hr"
        return f"{per_hour:.2f}/hr ~1 per {h:.1f}h"

    for _, row in summary_df.iterrows():
        drive_h = float(row.get("drive_hours", 0))
        peer = {}
        if fleet_avgs:
            for metric, key in [
                ("phone",         "phone_per_hour"),
                ("seatbelt",      "seatbelt_per_hour"),
                ("fatigue",       "fatigue_per_hour"),
                ("harsh",         "harsh_per_hour"),
                ("distraction",   "distraction_per_hour"),
                ("smoking",       "smoking_per_hour"),
                ("short_speeding","short_speeding_per_hour"),
                ("long_speeding", "long_speeding_per_hour"),
            ]:
                avg = fleet_avgs.get(key, 0)
                val = float(row.get(key, 0))
                peer[f"{metric}_vs_fleet"] = f"{round(val/avg, 1)}x fleet avg" if avg > 0 else "N/A"

        # Column name mapping from planner names to summary_df column names
        col_map = {
            "rank"            : "fleet_rank",
            "risk_class"      : "vehicle_behaviour_class",
            "ubi_score"       : "ubi_proxy_score",
            "drive_hours"     : "drive_hours",
            "distance_km"     : "distance_km",
            "crashes"         : "crash_count",
            "phone/hr"        : "phone_per_hour",
            "fatigue/hr"      : "fatigue_per_hour",
            "harsh/hr"        : "harsh_per_hour",
            "short_speeding/hr": "short_speeding_per_hour",
            "long_speeding/hr" : "long_speeding_per_hour",
            "distraction/hr"  : "distraction_per_hour",
            "smoking/hr"      : "smoking_per_hour",
            "seatbelt/hr"     : "seatbelt_per_hour",
            "power_violations": "power_violations",
            "situational_risk": "situational_risk",
            "cam_covered_detections": "cam_covered_detections",
            "covered_ratio"   : "covered_ratio",
            "cam_covered_review": "cam_covered_review_detections",
            "total_clips"     : "total_clips",
            "confidence"      : "evidence_confidence",
        }

        record = {
            "vehicle_id"  : int(row["vehicle_id"]),
            "fleet_rank"  : f"{int(row['fleet_rank'])} of {fleet_size} (1=most dangerous)",
            "risk_class"  : str(row["vehicle_behaviour_class"]),
            "ubi_score"   : float(row["ubi_proxy_score"]),
            "drive_hours" : float(row.get("drive_hours", 0)),
            "distance_km" : float(row.get("distance_km", 0)),
            "crashes"     : int(row.get("crash_count", 0)),
            "phone"       : to_human(float(row.get("phone_per_hour", 0))),
            "fatigue"     : to_human(float(row.get("fatigue_per_hour", 0))),
            "harsh"       : to_human(float(row.get("harsh_per_hour", 0))),
            "short_speeding" : to_human(float(row.get("short_speeding_per_hour", 0))),
            "long_speeding"  : to_human(float(row.get("long_speeding_per_hour", 0))),
            "distraction" : to_human(float(row.get("distraction_per_hour", 0))),
            "smoking"     : to_human(float(row.get("smoking_per_hour", 0))),
            "seatbelt"    : to_human(float(row.get("seatbelt_per_hour", 0))),
            "power_violations"      : int(row.get("power_violations", 0)),
            "situational_risk"      : int(row.get("situational_risk", 0)),
            "cam_covered_detections": int(row.get("cam_covered_detections", 0)),
            "covered_ratio"         : f"{float(row.get('covered_ratio', 0)):.1%}",
            "cam_covered_review"    : int(row.get("cam_covered_review_detections", 0)),
            "total_clips"           : int(row.get("total_clips", 0)),
            "evidence_confidence"   : str(row.get("evidence_confidence", "UNKNOWN")),
        }
        
        
        if peer:
            record["vs_fleet_avg"] = peer

        # Filter to planner-selected columns if provided
        if columns:
            allowed_keys = {"vehicle_id", "fleet_rank", "vs_fleet_avg"}
            for planner_col in columns:
                mapped = col_map.get(planner_col)
                if mapped:
                    allowed_keys.add(planner_col)
            record = {k: v for k, v in record.items()
                      if k in allowed_keys or k == "fleet_rank"}

        records.append(record)
    return json.dumps(records, indent=2)

def build_planner_summary(summary_df: pd.DataFrame) -> str:
    """
    Compact per-vehicle summary for planner LLM.
    One line per vehicle, /hr rates only, no V prefix.
    """
    lines = []
    for _, row in summary_df.iterrows():
        lines.append(
            f"{int(row['vehicle_id'])}: "
            f"rank={int(row['fleet_rank'])} "
            f"class={row['vehicle_behaviour_class']} "
            f"hours={float(row['drive_hours']):.2f} "
            f"crashes={int(row['crash_count'])} "
            f"phone={float(row.get('phone_per_hour', 0)):.3f}/hr "
            f"fatigue={float(row.get('fatigue_per_hour', 0)):.3f}/hr "
            f"harsh={float(row.get('harsh_per_hour', 0)):.3f}/hr "
            f"short_spd={float(row.get('short_speeding_per_hour', 0)):.3f}/hr "
            f"long_spd={float(row.get('long_speeding_per_hour', 0)):.3f}/hr "
            f"distraction={float(row.get('distraction_per_hour', 0)):.3f}/hr "
            f"smoking={float(row.get('smoking_per_hour', 0)):.3f}/hr "
            f"seatbelt={float(row.get('seatbelt_per_hour', 0)):.3f}/hr "
            f"power_viol={int(row.get('power_violations', 0))} "
            f"situational={int(row.get('situational_risk', 0))} "
            f"cam_covered={int(row.get('cam_covered_detections', 0))} "
            f"cam_review={int(row.get('cam_covered_review_detections', 0))} "
            f"covered_ratio={float(row.get('covered_ratio', 0)):.1%} "
            f"clips={int(row.get('total_clips', 0))} "
            f"conf={row['evidence_confidence']}"
        )
    return "\n".join(lines)

def execute_planner_tools(plan: dict, summary_df: pd.DataFrame, fleet_avgs: dict) -> dict:
    """
    Executes what the planner requested.
    New planner shape: focus_vehicles, fetch_clips.
    Sends focused JSON to main LLM — full fleet overview always included.
    """
    results = {
        "fleet_overview"   : "",
        "full_profiles"    : "",
        "clip_evidence"    : "",
        "clips_to_display" : [],
    }

    # ── Build focused data — planner selected vehicles and columns only ────────
    focus_ids = [int(v) for v in plan.get("focus_vehicles", [])]
    columns   = plan.get("columns", [])

    if not focus_ids:
        # Planner returned nothing — fallback to top 5 by rank
        focus_ids = summary_df.sort_values("fleet_rank").head(5)["vehicle_id"].tolist()

    focus_df = summary_df[summary_df["vehicle_id"].isin(focus_ids)].copy()

    if not focus_df.empty:
        results["full_profiles"] = summary_to_text(
            focus_df, fleet_avgs=fleet_avgs, columns=columns
        )

    # ── Fetch clips ────────────────────────────────────────────────────────────
    clip_requests = plan.get("fetch_clips", [])
    clip_lines    = []

    flag_map = {
        "phone"      : "phone_flag",
        "fatigue"    : "fatigue_flag",
        "cam_covered": "cam_covered_flag",
        "seatbelt"   : "seatbelt_off_flag",
        "smoking"    : "smoke_flag",
        "distraction": "distraction_flag",
    }
    no_clip_behaviours = {"harsh", "speeding", "short_speeding", "long_speeding",
                          "power_violations", "situational", "general"}

    for req in clip_requests:
        vid       = int(req.get("vehicle_id"))
        behaviour = str(req.get("behaviour", "")).lower().strip()
        max_clips = int(req.get("max", 5))

        if behaviour in no_clip_behaviours or behaviour not in flag_map:
            clip_lines.append(
                f"Vehicle {vid} | {behaviour}: no VLM clips exist for this behaviour."
            )
            continue

        flag     = flag_map[behaviour]
        clips_df = get_behaviour_clips(vid, flag).head(max_clips)

        if clips_df.empty:
            clip_lines.append(f"Vehicle {vid} | {behaviour}: no clips found.")
            continue

        clip_lines.append(f"\nVehicle {vid} | {behaviour} clips:")
        for _, r in clips_df.iterrows():
            clip_no = int(r["clip"])
            ts      = str(r.get("event_ts_final", ""))[:19]
            why     = str(r.get("why", ""))[:80]
            clip_lines.append(f"  clip_{clip_no:03d} | {ts} | {why}")
            results["clips_to_display"].append({
                "clip_no"   : clip_no,
                "vehicle_id": vid,
                "behaviour" : behaviour,
                "ts"        : ts,
                "why"       : why,
            })

    results["clip_evidence"] = "\n".join(clip_lines)
    return results

def build_planner_prompt(question: str, planner_summary: str, fleet_size: int,
                         camera_prefiltered: bool = False) -> str:

    return f"""You are a fleet safety data routing agent. A manager asked a question about their {fleet_size} drivers.
Your job is to select exactly which drivers and which columns the main analyst needs to answer the question well.
The main analyst will ONLY receive what you select — nothing else. Be precise and confident.

SCORING WEIGHTS (rank=1 is most dangerous):
  long_speeding=0.18, harsh=0.20, short_speeding=0.12, distraction/phone/seatbelt/smoking=0.13,
  fatigue=0.12, power_violations=0.10, situational=0.10, camera_covering=0.05
  crashes and drive_hours do not affect rank but give important context.
  cam_covered_ratio: % of clips where camera was confirmed covered. Always read alongside total_clips. For perspective.

BEHAVIOURS WITH VLM CLIPS (only these can be requested):
  phone, fatigue, cam_covered, smoking, seatbelt, distraction
  DO NOT request clips for: harsh, speeding, power_violations, situational — no dashcam clips exist for these.
  Only request clips if the manager specifically asked for evidence, investigation or formal action on one of the above behaviours.
  One behaviour per clip request only.

LARGE FLEET OVERVIEW RULE: If fleet > 10 and the question asks for a general overview or executive summary:
  - Set focus_vehicles to ALL vehicle IDs
  - Select only these columns:drive_hours, distance_km, crashes,short_speeding/hr, long_speeding/hr,distraction/hr,fatigue/hr,power_violations, situational_risk,harsh/hr,covered_ratio,total_clips.
  - Do NOT request clips
  - The main analyst will identify outliers from the full dataset

FLEET SUMMARY:
{planner_summary}

MANAGER QUESTION: {question}

Decide:
1. Which drivers are most relevant? For specific behaviour questions pick the worst offenders. For small fleets pick all. For large overview pick all but limit columns.
2. Which columns does the main analyst need? Pick only what is relevant to answer the question.
3. Clips? Only if manager asked for evidence/investigation on: phone, fatigue, cam_covered, smoking, seatbelt or distraction. One behaviour per request.

Respond ONLY in valid JSON. No prose. Integer IDs only.

{{
  "focus_vehicles": [list of vehicle_ids],
  "columns": [list of column names needed from: rank, risk_class, ubi_score, drive_hours, distance_km, crashes, phone/hr, fatigue/hr, harsh/hr, short_speeding/hr, long_speeding/hr, distraction/hr, smoking/hr, seatbelt/hr, power_violations, situational_risk, cam_covered_detections, covered_ratio, cam_covered_review, total_clips, confidence],
  "fetch_clips": [
    {{"vehicle_id": 123456, "behaviour": "phone|fatigue|cam_covered|smoking|seatbelt|distraction", "max": 5}}
  ],
  "reasoning": "one sentence"
}}"""

def plot_risk_heatmap(summary_df: pd.DataFrame, title: str = "Fleet Risk Heatmap"):
    """Behaviour vs vehicle heatmap — colour coded by normalised severity."""
    behaviours = {
        "Phone/hr"    : "phone_per_hour",
        "Seatbelt/hr" : "seatbelt_per_hour",
        "Fatigue/hr"  : "fatigue_per_hour",
        "Harsh/hr"    : "harsh_per_hour",
        "Cam covered%": "covered_ratio",
        "Crashes"     : "crash_count",
    }
    available = {k: v for k, v in behaviours.items() if v in summary_df.columns}
    if not available:
        return

    # Top 20 by fleet rank for readability
    plot_df = summary_df.head(20).copy()
    matrix  = plot_df[[v for v in available.values()]].copy()

    # Normalise each column 0-1
    for col in matrix.columns:
        col_max = matrix[col].max()
        matrix[col] = matrix[col] / col_max if col_max > 0 else 0

    matrix.index = [f"V{int(v)}" for v in plot_df["vehicle_id"]]
    matrix.columns = list(available.keys())

    fig, ax = plt.subplots(figsize=(len(available) * 1.5 + 2, len(plot_df) * 0.4 + 2))
    im = ax.imshow(matrix.values, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=30, ha="right", fontsize=9)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index, fontsize=8)
    plt.colorbar(im, ax=ax, label="Relative severity (0=low, 1=highest in fleet)")
    ax.set_title(f"{title}\n(Top 20 by risk rank — red = worst)", fontsize=11)
    plt.tight_layout()
    plt.show()

def plot_daily_trend(vehicle_id: int, daily_df: pd.DataFrame):
    """Daily violation pattern for a single vehicle."""
    if daily_df.empty:
        print(f"  No daily data for vehicle {vehicle_id}")
        return
    cols = ["phone_detections","seatbelt_violations","fatigue_detections","cam_covered"]
    cols = [c for c in cols if c in daily_df.columns]
    fig, ax = plt.subplots(figsize=(12, 4))
    for col in cols:
        ax.plot(daily_df["day"].astype(str), daily_df[col],
                marker="o", linewidth=1.5, label=col.replace("_", " "))
    ax.set_title(
        f"Vehicle {vehicle_id} — Daily Violation Pattern\n"
        f"(7-day period — trend indicative only, longer period needed for significance)",
        fontsize=10
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Detections")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert fleet safety analyst. You write clear, professional responses for fleet managers, executives and HR.
You receive driver data as JSON. All rates are per hour with human-readable equivalents — always use the human-readable form (≈1 per Xh) in your response, never raw /hr numbers alone.

═══ RANKING ═══
rank=1 is the most dangerous driver. Lower rank = more dangerous. Higher rank = safer.
Rank is authoritative — it combines all weighted behaviours. Never override it based on a single metric.
If a driver has an extreme outlier metric but a lower rank, explain why.

═══ SCORING WEIGHTS ═══
long_speeding=0.18, harsh=0.20, short_speeding=0.12, distraction/phone/seatbelt/smoking=0.13,
fatigue=0.12, power_violations=0.10, situational=0.10, camera_covering=0.05
crashes and drive_hours do not affect rank but always provide context.
Low drive hours = less reliable rating — flag this if in context.
cam_covered_ratio: % of clips where camera was confirmed covered. Always read alongside total_clips. For perspective.
You only receive the data the planner selected. Base your answer entirely on what is provided.

Answer the question directly and naturally. Be specific — use vehicle IDs and exact figures.
Use ≈ for approximations. Professional tone throughout.
If clips are provided, reference specific clip numbers as evidence.
If the question is about a specific behaviour, lead with the worst offender for that behaviour."""

# QUERY ROUTER

In [ ]:
# =============================================================================
# CELL 5 — QUERY ROUTER (agentic two-step LLM pipeline)
# =============================================================================
# Step 1: Small/fast planner LLM decides what data + clips the main LLM needs
# Step 2: Main LLM receives exactly that data and writes the executive answer
# Step 3: Python displays clips only if the main LLM's plan requested them
# =============================================================================

import json as _json

PLANNER_MODEL     =  "qwen2.5:14b"  # fast planner — swap to 14b if 3b not available "qwen2.5:14b" ("qwen2.5:3b" was too stupid)
PLANNER_MAX_TOKENS = 400

def _call_planner(prompt: str) -> tuple[dict, float]:
    """Call planner LLM and parse JSON response. Strips V prefix from vehicle IDs."""
    payload = {
        "model"  : PLANNER_MODEL,
        "prompt" : prompt,
        "stream" : False,
        "options": {"temperature": 0.0, "seed": 42, "num_predict": PLANNER_MAX_TOKENS},
    }
    t0 = time.perf_counter()
    r  = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, timeout=1500)
    elapsed = time.perf_counter() - t0
    r.raise_for_status()
    raw = r.json().get("response", "").strip()

    # Strip markdown fences
    clean = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
    clean = re.sub(r"\s*```$",          "", clean, flags=re.MULTILINE).strip()

    # Extract first complete JSON object
    brace_count, end_idx = 0, None
    for i, ch in enumerate(clean):
        if ch == "{":
            brace_count += 1
        elif ch == "}":
            brace_count -= 1
            if brace_count == 0:
                end_idx = i + 1
                break
    if end_idx:
        clean = clean[:end_idx]

    # Strip V prefix from vehicle IDs inside the JSON string
    clean = re.sub(r'"V(\d+)"', r'\1', clean)
    clean = re.sub(r'V(\d+)',   r'\1', clean)

    try:
        plan = _json.loads(clean)
    except Exception as e:
        print(f"  ⚠ Planner JSON parse failed: {e}")
        print(f"  Raw response: {raw[:400]}")
        plan = {
            "summary_filter" : "top_n",
            "filter_by"      : "fleet_rank",
            "filter_n"       : 5,
            "enrich_vehicles": [],
            "fetch_clips"    : [],
            "reasoning"      : "fallback — JSON parse failed",
        }

    return plan, elapsed


# ── Debug flag — set False to silence all debug output ────────────────────────
DEBUG_LLM = False

def run_query(
    question       : str,
    vehicle_ids    : list,
    max_tokens     : int  = 2500,
    auto_show_clips: bool = True,
) -> None:
    ids     = [int(v) for v in vehicle_ids]
    t_start = time.perf_counter()

    print(f"\n{'='*70}")
    print(f"FLEET SIZE   : {len(ids)} vehicles")
    print(f"QUESTION     : {question}")
    print(f"{'='*70}\n")

    summary    = get_vehicle_summary(ids)
    fleet_avgs = get_fleet_averages(summary)

    if summary.empty:
        print("No data found for these vehicle IDs.")
        return

    # ── Full rankings table ───────────────────────────────────────────────────
    print("FULL FLEET RANKINGS")
    print("─" * 70)
    display(summary[[
        "vehicle_id", "fleet_rank", "vehicle_behaviour_class", "ubi_proxy_score",
        "drive_hours", "distance_km", "crash_count",
        "phone_per_hour", "fatigue_per_hour", "harsh_per_hour",
        "short_speeding_per_hour", "long_speeding_per_hour",
        "distraction_per_hour", "smoking_per_hour",
        "seatbelt_per_hour", "power_violations", "situational_risk",
        "cam_covered_detections", "covered_ratio", "cam_covered_review_detections",
        "total_clips", "evidence_confidence",
    ]])
    print("─" * 70 + "\n")

    vehicles_no_clips = sum(1 for v in ids if v not in fused_event_df["vehicle_id"].values)
    coverage_note = ""
    if vehicles_no_clips > 0:
        coverage_note = (
            f"\nNOTE: {vehicles_no_clips} of {len(ids)} vehicles have NO camera footage "
            f"— they cannot be visually assessed.\n"
        )

    # ── Camera pre-filter ─────────────────────────────────────────────────────
    camera_keywords       = ["cover", "camera", "tamper", "obstruct", "block"]
    is_camera_question    = any(kw in question.lower() for kw in camera_keywords)
    camera_prefiltered    = False
    planner_input_summary = summary.copy()

    if is_camera_question:
        covered = summary[
            (summary["cam_covered_detections"] > 0) |
            (summary["cam_covered_review_detections"] > 0)
        ].copy()
        if not covered.empty:
            planner_input_summary = covered
            camera_prefiltered    = True
    # ── Step 1: Planner ───────────────────────────────────────────────────────
    print("Step 1: Planning query...")
    planner_summary = build_planner_summary(planner_input_summary)
    planner_prompt  = build_planner_prompt(question, planner_summary, len(ids),
                                           camera_prefiltered=camera_prefiltered)

    if DEBUG_LLM:
        print(f"\n{'─'*40} DEBUG: SYSTEM PROMPT {'─'*40}")
        print(SYSTEM_PROMPT)
        print(f"{'─'*70}\n")
        print(f"\n{'─'*40} DEBUG: PLANNER PROMPT {'─'*40}")
        print(planner_prompt)
        print(f"{'─'*70}\n")

    plan, t_planner = _call_planner(planner_prompt)

    print(f"  Focus vehicles : {plan.get('focus_vehicles')}")
    print(f"  Clips          : {plan.get('fetch_clips')}")
    print(f"  Reasoning      : {plan.get('reasoning')}")
    print(f"  Planner time   : {t_planner:.1f}s\n")

    if DEBUG_LLM:
        print(f"\n{'─'*40} DEBUG: PLANNER RAW PLAN {'─'*40}")
        print(json.dumps(plan, indent=2))
        print(f"{'─'*70}\n")

    # ── Step 2: Execute tools ─────────────────────────────────────────────────
    tool_results = execute_planner_tools(plan, summary, fleet_avgs)

    # ── Step 3: Build main LLM prompt ─────────────────────────────────────────
    parts = [
        f"Manager question: {question}\n",
        coverage_note,
        f"Fleet size: {len(ids)} vehicles\n",
    ]

    if tool_results["full_profiles"]:
        parts.append(f"\nDRIVER DATA:\n{tool_results['full_profiles']}\n")
    if tool_results["clip_evidence"]:
        parts.append(f"\nCLIP EVIDENCE:\n{tool_results['clip_evidence']}\n")

    main_prompt = "\n".join(parts)

    if DEBUG_LLM:
        print(f"\n{'─'*40} DEBUG: MAIN LLM PROMPT {'─'*40}")
        print(main_prompt[:6000])
        if len(main_prompt) > 6000:
            print(f"... [truncated — total length: {len(main_prompt)} chars]")
        print(f"{'─'*70}\n")

    # ── Step 4: Main LLM ──────────────────────────────────────────────────────
    print(f"{'─'*70}")
    print("Step 2: Building answer...")
    print(f"{'─'*70}\n")

    response, t_llm = ask_llm(main_prompt, system=SYSTEM_PROMPT, max_tokens=max_tokens)

    if DEBUG_LLM:
        print(f"\n{'─'*40} DEBUG: RAW LLM RESPONSE {'─'*40}")
        print(response[:3000])
        if len(response) > 3000:
            print(f"... [truncated — total length: {len(response)} chars]")
        print(f"{'─'*70}\n")

    print(f"\n{'='*70}")
    print(f"FLEET ANALYSIS — {question[:60]}  [{t_llm:.1f}s]")
    print(f"{'='*70}")
    display(Markdown(response))

    # ── Step 5: Clips ─────────────────────────────────────────────────────────
    clips_to_show = tool_results["clips_to_display"]

    if clips_to_show:
        if auto_show_clips:
            print(f"\n{'─'*70}")
            print("SUPPORTING VISUAL EVIDENCE")
            print(f"{'─'*70}")
            for c in clips_to_show:
                print(f"\nclip_{c['clip_no']:03d} | {c['vehicle_id']} | "
                      f"{c['behaviour']} | {c['ts']} | {c['why']}")
                show_clip_frames(
                    c["clip_no"],
                    title=f"{c['vehicle_id']} | clip_{c['clip_no']:03d} | {c['behaviour']} | {c['ts']}",
                    max_frames=12,
                )
        else:
            mentioned = {int(m) for m in re.findall(r"clip_(\d+)", response)}
            shown     = [c for c in clips_to_show if c["clip_no"] in mentioned]
            if shown:
                print(f"\n{'─'*70}")
                print("SUPPORTING VISUAL EVIDENCE (referenced by analyst)")
                print(f"{'─'*70}")
                for c in shown:
                    print(f"\nclip_{c['clip_no']:03d} | {c['vehicle_id']} | "
                          f"{c['behaviour']} | {c['ts']}")
                    show_clip_frames(
                        c["clip_no"],
                        title=f"{c['vehicle_id']} | clip_{c['clip_no']:03d} | {c['behaviour']} | {c['ts']}",
                        max_frames=12,
                    )
                    

    t_total = time.perf_counter() - t_start
    print(f"\n⏱ Planner time  : {t_planner:.1f}s")
    print(f"⏱ LLM time      : {t_llm:.1f}s")
    print(f"⏱ Total         : {t_total:.1f}s")
    print(f"   Model: {OLLAMA_MODEL} (local) | Hardware: Intel Core Ultra 7 155H, 32GB RAM")

print("✓ Cell 5 loaded — agentic two-step pipeline ready")

In [ ]:
# =============================================================================
# MANAGER A — Driver Summary Table
# =============================================================================

manager_a_ids = [507451763, 312741040]

manager_a_summary = get_vehicle_summary(manager_a_ids)

cols  = [
    "vehicle_id", "fleet_rank", "ubi_proxy_score",
    "drive_hours", "distance_km", "crash_count",
    "phone_per_hour", "fatigue_per_hour", "harsh_per_hour",
    "short_speeding_per_hour", "long_speeding_per_hour",
    "distraction_per_hour",
    "seatbelt_per_hour", "power_violations", "situational_risk",
    "cam_covered_detections", "covered_ratio",
    "total_clips",
]

pivoted = manager_a_summary[cols].set_index("vehicle_id").T
pivoted

#  MANAGER A

In [ ]:
# =============================================================================
# MANAGER A — Fleet of 3, driver ratings
# =============================================================================

random.seed(RANDOM_SEED)
manager_a_vehicles = random.sample(all_vehicle_ids, 3)

run_query(
    question    = "Give me a full safety assessment of my drivers. "
                  "Who is the worst and why? I need something I can use "
                  "in their performance review.",
    vehicle_ids = manager_a_vehicles,
    max_tokens  = 2000,
)

# MANAGER B

In [ ]:
# =============================================================================
# MANAGER B — Fleet of 50, camera covering
# =============================================================================

random.seed(RANDOM_SEED)
manager_b_vehicles = random.sample(all_vehicle_ids, 50)

run_query(
    question    = "Which of my drivers are covering their camera? "
                  "I want to know if it looks deliberate and what I should do.",
    vehicle_ids = manager_b_vehicles,
    max_tokens  = 2000,
)

# MANAGER C

In [ ]:
# =============================================================================
# MANAGER C — Fleet of 10, phone usage investigation
# Version A: auto_show_clips=True  — always show clips planner requested
# Version B: auto_show_clips=False — only show clips LLM references by name
# =============================================================================

random.seed(RANDOM_SEED + 1)
manager_c_vehicles = random.sample(all_vehicle_ids, 10)

print("VERSION A — auto show clips")
run_query(
    question        = "Investigate phone use across my drivers. "
                      "Who is the worst offender? I need evidence and a formal warning.",
    vehicle_ids     = manager_c_vehicles,
    max_tokens      = 2500,
    auto_show_clips = True,
)

print("\n" + "="*70)
print("VERSION B — LLM decides which clips to reference")
print("="*70 + "\n")

run_query(
    question        = "Investigate phone use across my drivers. "
                      "Who is the worst offender? I need evidence and a formal warning.",
    vehicle_ids     = manager_c_vehicles,
    max_tokens      = 2500,
    auto_show_clips = False,
)

# MANAGER D

In [ ]:
# =============================================================================
# MANAGER D — Fleet of 100, exec overview
# =============================================================================

random.seed(RANDOM_SEED + 2)
manager_d_vehicles = random.sample(all_vehicle_ids, min(100, len(all_vehicle_ids)))

run_query(
    question    = "Give me an overview of my entire fleet. "
                  "I need to present this to the executive team to get budget "
                  "approved for driver safety training.",
    vehicle_ids = manager_d_vehicles,
    max_tokens  = 3500,
)